In [ ]:

import re
import numpy as np
import faiss
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder


EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"


def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


class ResearchRetriever:
    def __init__(self, chunks):
        if not chunks:
            raise ValueError("At least one text chunk is required.")

        self.chunks = chunks
        self.bm25 = BM25Okapi([
            tokenize(chunk["text"]) for chunk in chunks
        ])

        self.device = (
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        self.embedder = SentenceTransformer(
            EMBED_MODEL,
            device=self.device
        )

        embeddings = self.embedder.encode(
            [chunk["text"] for chunk in chunks],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        embeddings = np.asarray(embeddings, dtype="float32")

        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)

        self._reranker = None

    def _format_results(self, indices, scores):
        results = []

        for idx, score in zip(indices, scores):
            if idx < 0:
                continue

            item = dict(self.chunks[int(idx)])
            item["score"] = float(score)
            results.append(item)

        return results

    def search_bm25(self, query, top_k=5):
        scores = np.asarray(
            self.bm25.get_scores(tokenize(query))
        )

        top_k = min(top_k, len(self.chunks))
        indices = np.argsort(scores)[::-1][:top_k]

        return self._format_results(indices, scores[indices])

    def search_dense(self, query, top_k=5):
        query_embedding = self.embedder.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        top_k = min(top_k, len(self.chunks))
        scores, indices = self.index.search(query_embedding, top_k)

        return self._format_results(indices[0], scores[0])

    def search_hybrid(self, query, top_k=5, candidate_k=10, rrf_k=60):
        bm25_results = self.search_bm25(query, candidate_k)
        dense_results = self.search_dense(query, candidate_k)

        # The document/page/chunk combination identifies a passage.
        ranked = {}

        for result_list in (bm25_results, dense_results):
            for rank, result in enumerate(result_list, start=1):
                key = result["chunk_id"]

                if key not in ranked:
                    ranked[key] = {
                        "result": result,
                        "rrf_score": 0.0
                    }

                ranked[key]["rrf_score"] += 1.0 / (rrf_k + rank)

        merged = sorted(
            ranked.values(),
            key=lambda item: item["rrf_score"],
            reverse=True
        )

        results = []

        for item in merged[:top_k]:
            result = dict(item["result"])
            result["score"] = item["rrf_score"]
            results.append(result)

        return results

    def rerank(self, query, candidates, top_k=5):
        if not candidates:
            return []

        if self._reranker is None:
            self._reranker = CrossEncoder(
                RERANK_MODEL,
                device=self.device
            )

        pairs = [
            (query, candidate["text"])
            for candidate in candidates
        ]

        scores = self._reranker.predict(pairs)

        reranked = []

        for candidate, score in zip(candidates, scores):
            item = dict(candidate)
            item["rerank_score"] = float(score)
            reranked.append(item)

        reranked.sort(
            key=lambda item: item["rerank_score"],
            reverse=True
        )

        return reranked[:top_k]
